In [3]:
!pip install torch torchvision matplotlib tqdm

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np


In [5]:
# Config
IMG_SIZE = 64
BATCH_SIZE = 32
epoch = 20
LEARNING_RATE = 2e-4
T = 300
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cpu


In [9]:
import time
import torch
import platform
import multiprocessing

start = time.time()
dummy = torch.randn(64, 3, 64, 64)
for i in range(100):
    _ = dummy * 3.14 + 2.71
end = time.time()

print(" CPU Test Completed")
print(f"Time taken for 100 ops on 64x64x3 batch: {end - start:.4f} seconds")
print(f"CPU Name: {platform.processor()}")
print(f"Cores available: {multiprocessing.cpu_count()}")


 CPU Test Completed
Time taken for 100 ops on 64x64x3 batch: 0.0881 seconds
CPU Name: arm
Cores available: 8


In [10]:
# Transform
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    #transforms.Normalize([0.5], [0.5])
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])
root_path = "./final_processed_data_2/train"  # Change this if your data is elsewhere

# List files in root_path for sanity check
print("Files/folders in dataset path:")
print(os.listdir(root_path))

# Dataset
dataset = ImageFolder("./final_processed_data_2/train", transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)



# Sanity check
print("Number of samples:", len(dataset))
print("Num samples in dataset:", len(dataset))
print("Num batches per epoch:", len(dataloader))


# # Print the first 5 batch shapes for confirmation
# for batch_idx, (images, labels) in enumerate(dataloader):
#     print(f"Batch {batch_idx+1} | Images shape: {images.shape}")
#     if batch_idx == 4:
#         break


Files/folders in dataset path:
['.DS_Store', 'outside', 'inside', 'food', 'drink']
Number of samples: 18599
Num samples in dataset: 18599
Num batches per epoch: 582


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim=None):
        super().__init__()
        self.norm = nn.BatchNorm2d(in_ch)
        self.act = nn.SiLU()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_emb = None
        if time_emb_dim is not None:
            self.time_emb = nn.Linear(time_emb_dim, out_ch)
    def forward(self, x, t=None):
        h = self.act(self.norm(x))
        h = self.conv(h)
        if self.time_emb is not None and t is not None:
            # t shape: [batch, time_emb_dim]
            h += self.time_emb(t)[:, :, None, None]
        return h

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.block1 = Block(in_ch, out_ch, time_emb_dim)
        self.block2 = Block(out_ch, out_ch, time_emb_dim)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x, t):
        h = self.block1(x, t)
        h = self.block2(h, t)
        return self.pool(h), h

class Up(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.block1 = Block(in_ch, out_ch, time_emb_dim)
        self.block2 = Block(out_ch, out_ch, time_emb_dim)
    def forward(self, x, skip, t):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = torch.cat([x, skip], dim=1)
        h = self.block1(x, t)
        h = self.block2(h, t)
        return h

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.fc1 = nn.Linear(1, dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim)
    def forward(self, t):
        t = t.float().unsqueeze(-1) / 1000  # [batch] -> [batch, 1]
        t = self.act(self.fc1(t))
        t = self.fc2(t)
        return t

class StrongerUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, time_emb_dim=256):
        super().__init__()
        self.time_mlp = TimeEmbedding(time_emb_dim)
        # Encoder
        self.down1 = Down(in_ch, 64, time_emb_dim)
        self.down2 = Down(64, 128, time_emb_dim)
        self.down3 = Down(128, 256, time_emb_dim)
        # Middle
        self.mid_block1 = Block(256, 256, time_emb_dim)
        self.mid_block2 = Block(256, 256, time_emb_dim)
        # Decoder
        self.up3 = Up(512, 128, time_emb_dim)  # 256+256
        self.up2 = Up(256, 64, time_emb_dim)   # 128+128
        self.up1 = Up(128, 32, time_emb_dim)   # 64+64
        self.final_conv = nn.Conv2d(32, out_ch, 1)
    
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        # Downsampling
        h1, skip1 = self.down1(x, t_emb)
        h2, skip2 = self.down2(h1, t_emb)
        h3, skip3 = self.down3(h2, t_emb)
        # Middle
        mid = self.mid_block1(h3, t_emb)
        mid = self.mid_block2(mid, t_emb)
        # Upsampling
        up3 = self.up3(mid, skip3, t_emb)
        up2 = self.up2(up3, skip2, t_emb)
        up1 = self.up1(up2, skip1, t_emb)
        out = self.final_conv(up1)
        return out

    #output = model(dummy_image, dummy_t)
    #print("Model ran successfully.")
   #print("Input shape:", dummy_image.shape)
    #print("Output shape:", output.shape)


In [12]:
# CONFIG
T = 300
LEARNING_RATE = 2e-4
IMG_SIZE = 64
epoch = 20
BATCH_SIZE = 32
device = torch.device("cpu")



# Noise scheduler 
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1. - betas
alpha_bars = torch.cumprod(alphas, dim=0).to(device)

model = StrongerUNet().to(device)
# Optimizer and loss 
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

# Forward diffusion function 
def forward_diffusion(x_0, t):
    noise = torch.randn_like(x_0).to(device)
    sqrt_alpha_bar = alpha_bars[t][:, None, None, None].sqrt()
    sqrt_one_minus = (1 - alpha_bars[t])[:, None, None, None].sqrt()
    return sqrt_alpha_bar * x_0 + sqrt_one_minus * noise, noise

    #  Test forward_diffusion() function
with torch.no_grad():
    x0 = torch.randn(4, 3, 64, 64).to(device)  # batch of clean images
    t = torch.randint(0, T, (4,), device= device)  # random timesteps for each image

    x_t, noise = forward_diffusion(x0, t)

    print(" forward_diffusion() ran successfully.")
    print("Input x0 shape:  ", x0.shape)
    print("Output x_t shape:", x_t.shape)
    print("Noise shape:     ", noise.shape)
    print("Sample timesteps:", t.tolist())
    print("Pixel range in x_t: [{:.4f}, {:.4f}]".format(x_t.min().item(), x_t.max().item()))



 forward_diffusion() ran successfully.
Input x0 shape:   torch.Size([4, 3, 64, 64])
Output x_t shape: torch.Size([4, 3, 64, 64])
Noise shape:      torch.Size([4, 3, 64, 64])
Sample timesteps: [179, 248, 138, 157]
Pixel range in x_t: [-4.7442, 4.1232]


In [13]:
def q_sample(x0, t, noise):
    T = 300
    betas = torch.linspace(1e-4, 0.02, T).to(x0.device)
    alphas = 1. - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    a_bar = alpha_bars[t].view(-1, 1, 1, 1)
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise

    #  Test q_sample() function
    with torch.no_grad():
        x0 = torch.randn(4, 3, 64, 64).to(device)  # batch of clean images
        noise = torch.randn_like(x0).to(device)    # random noise
        t = torch.randint(0, 300, (4,), device=device)  # random timesteps for each image in batch

    x_t = q_sample(x0, t, noise)  # apply noise

    print(" q_sample() ran successfully.")
    print("Original x0 shape:", x0.shape)
    print("Noisy x_t shape:   ", x_t.shape)
    print("Sample timestep t: ", t.tolist())



In [9]:
# print("DEBUG concat shape:", x.shape)


In [14]:
def ddim_sample(model, x, T=300):
    model.eval()
    betas = torch.linspace(1e-4, 0.02, T).to(x.device)
    alphas = 1. - betas
    alpha_bars = torch.cumprod(alphas, dim=0)

    for t in reversed(range(T)):
        t_batch = torch.tensor([t], dtype=torch.long).to(x.device)

        eps_theta = model(x, t_batch)

        alpha_bar_t = alpha_bars[t]
        alpha_bar_t_sqrt = torch.sqrt(alpha_bar_t)
        one_minus_alpha_bar_sqrt = torch.sqrt(1 - alpha_bar_t)

        x0_pred = (x - one_minus_alpha_bar_sqrt * eps_theta) / alpha_bar_t_sqrt

        if t > 0:
            alpha_bar_prev = alpha_bars[t - 1]
            sigma = 0
            noise = torch.randn_like(x) if sigma > 0 else 0

            x = (
                torch.sqrt(alpha_bar_prev) * x0_pred +
                torch.sqrt(1 - alpha_bar_prev - sigma**2) * eps_theta +
                sigma * noise
            )
        else:
            x = x0_pred

    return x
#  Test ddim_sample() function
with torch.no_grad():
    noise = torch.randn(1, 3, 64, 64).to(device)  # start from pure noise
    sampled = ddim_sample(model, noise, T=300)
    sampled = (sampled + 1) / 2      # change from [-1, 1] to [0, 1]
    sampled = sampled.clamp(0, 1)    # cut off anything outside [0, 1]


    print("ddim_sample() ran successfully.")
    print("Input noise shape:  ", noise.shape)
    print("Sampled image shape:", sampled.shape)
    print("Pixel value range:  [{:.4f}, {:.4f}]".format(sampled.min().item(), sampled.max().item()))


ddim_sample() ran successfully.
Input noise shape:   torch.Size([1, 3, 64, 64])
Sampled image shape: torch.Size([1, 3, 64, 64])
Pixel value range:  [0.0000, 1.0000]


In [15]:
def save_sample_images(model, epoch):
    model.eval()
    with torch.no_grad():
        noise = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        sampled = ddim_sample(model, noise)

        # Fix pixel range for saving
        sampled = (sampled + 1) / 2
        sampled = sampled.clamp(0, 1)

        # Save the image
        os.makedirs("results", exist_ok=True)
        save_image(sampled, f"results/sample_epoch_{epoch}.png")

        # Print success message
        print(f" Sample image saved: results/sample_epoch_{epoch}.png")
        print(f"  Shape: {sampled.shape}, Pixel range: [{sampled.min():.4f}, {sampled.max():.4f}]")
        
#  Run test
save_sample_images(model, epoch=0)

 Sample image saved: results/sample_epoch_0.png
  Shape: torch.Size([1, 3, 64, 64]), Pixel range: [0.0000, 1.0000]


In [ ]:
import torch
from tqdm import tqdm

epoch = 50
for epoch in range(1, epoch + 1):  # e.g., 50 epochs
    model.train()
    running_loss = 0.0
    num_batches = 0
    progress = tqdm(dataloader, desc=f"epoch {epoch}/{epoch}")

    for images, _ in progress:
        images = images.to(device)
        t = torch.randint(0, T, (images.size(0),), device=device).long()
        noise = torch.randn_like(images)  # generate random noise
        x_t = q_sample(images, t, noise)  # make noisy images


        predicted_noise = model(x_t, t)
        loss = loss_fn(predicted_noise, noise)

        #loss = F.mse_loss(predicted_noise, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        progress.set_postfix({"loss": loss.item()})

    avg_loss = running_loss / num_batches
    print(f"epoch {epoch} — Loss: {avg_loss:.4f}")

    # Save model and generate sample images every 5 epochs
    if epoch % 5 == 0:
        torch.save(model.state_dict(), f"checkpoints/ddim_epoch_{epoch}.pth")
        save_sample_images(model, epoch)


epoch 1/1: 100%|██████████| 582/582 [15:42<00:00,  1.62s/it, loss=0.192]


epoch 1 — Loss: 0.2477


epoch 2/2: 100%|██████████| 582/582 [16:48<00:00,  1.73s/it, loss=0.193] 


epoch 2 — Loss: 0.1338


epoch 3/3: 100%|██████████| 582/582 [16:44<00:00,  1.73s/it, loss=0.111] 


...

epoch 43/43: 100%|██████████| 582/582 [14:57<00:00,  1.54s/it, loss=0.0666]


epoch 43 — Loss: 0.0521


epoch 44/44: 100%|██████████| 582/582 [15:09<00:00,  1.56s/it, loss=0.0311]


epoch 44 — Loss: 0.0518


epoch 45/45:  23%|██▎       | 134/582 [03:55<23:15,  3.11s/it, loss=0.0448]

In [25]:
model = StrongerUNet().to(device)
model.load_state_dict(torch.load('checkpoints/ddim_epoch_40.pth', map_location=device))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [26]:
# 1. Re-initialize your model and optimizer
model = StrongerUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Load the checkpoint (model weights)
checkpoint_epoch = 40
checkpoint = torch.load('checkpoints/ddim_epoch_40.pth', map_location=device)
#checkpoint_path = f"checkpoints/ddim_epoch_{checkpoint_epoch}.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))


<All keys matched successfully>

In [27]:
TOTAL_EPOCHS = 50  # or whatever number you want
for epoch in range(checkpoint_epoch + 1, TOTAL_EPOCHS + 1):
    # ---- your training code here ----
    # (same code as before)
    # At the end of each epoch, save a new checkpoint:
    torch.save(model.state_dict(), f"checkpoints/ddim_epoch_{epoch}.pth")
    # Optionally: save your sample images
    if epoch % 5 == 0:
        save_sample_images(model, epoch)


 Sample image saved: results/sample_epoch_45.png
  Shape: torch.Size([1, 3, 64, 64]), Pixel range: [0.0000, 1.0000]
 Sample image saved: results/sample_epoch_50.png
  Shape: torch.Size([1, 3, 64, 64]), Pixel range: [0.0000, 1.0000]


In [31]:
from tqdm import tqdm
import torch

TOTAL_EPOCHS = 50
device = torch.device('cpu')  # or 'cuda' if you have GPU

for epoch in range(45, TOTAL_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    print(f"\n=== EPOCH {epoch}/{TOTAL_EPOCHS} ===")
    num_batches = len(dataloader)
    
    # tqdm gives you a nice progress bar
    for batch_idx, (images, _) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch}"), 1):
        images = images.to(device)
        t = torch.randint(0, T, (images.size(0),), device=device)
        noise = torch.randn_like(images)
        noisy_images = q_sample(images, t, noise)

        output = model(noisy_images, t)
        loss = loss_fn(output, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        if batch_idx % 10 == 0 or batch_idx == num_batches:
            print(f"  Batch {batch_idx}/{num_batches} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / num_batches
    print(f"=== FINISHED EPOCH {epoch} | Avg loss: {avg_loss:.4f} ===")

    # Save every 5 epochs
    if epoch % 5 == 0:
        torch.save(model.state_dict(), f"checkpoints/ddim_epoch_{epoch}.pth")
        save_sample_images(model, epoch)



=== EPOCH 45/50 ===


Epoch 45:   2%|▏         | 10/582 [00:24<23:37,  2.48s/it]

  Batch 10/582 | Loss: 0.0430


Epoch 45:   3%|▎         | 20/582 [00:49<22:34,  2.41s/it]

  Batch 20/582 | Loss: 0.0591


...

Epoch 50: 100%|█████████▉| 580/582 [14:31<00:03,  1.62s/it]

  Batch 580/582 | Loss: 0.0460


Epoch 50: 100%|██████████| 582/582 [14:33<00:00,  1.50s/it]

  Batch 582/582 | Loss: 0.0612
=== FINISHED EPOCH 50 | Avg loss: 0.0515 ===


 Sample image saved: results/sample_epoch_50.png
  Shape: torch.Size([1, 3, 64, 64]), Pixel range: [0.0235, 1.0000]


In [ ]:
# for batch_idx, (images, _) in enumerate(dataloader):
#     print(f"Batch {batch_idx}: {images.shape}")
#     # ...rest of training...


Batch 0: torch.Size([32, 3, 64, 64])
Batch 1: torch.Size([32, 3, 64, 64])
Batch 2: torch.Size([32, 3, 64, 64])
Batch 3: torch.Size([32, 3, 64, 64])
Batch 4: torch.Size([32, 3, 64, 64])
Batch 5: torch.Size([32, 3, 64, 64])
Batch 6: torch.Size([32, 3, 64, 64])
Batch 7: torch.Size([32, 3, 64, 64])
Batch 8: torch.Size([32, 3, 64, 64])
Batch 9: torch.Size([32, 3, 64, 64])
Batch 10: torch.Size([32, 3, 64, 64])
Batch 11: torch.Size([32, 3, 64, 64])
Batch 12: torch.Size([32, 3, 64, 64])
Batch 13: torch.Size([32, 3, 64, 64])
Batch 14: torch.Size([32, 3, 64, 64])
Batch 15: torch.Size([32, 3, 64, 64])
Batch 16: torch.Size([32, 3, 64, 64])
Batch 17: torch.Size([32, 3, 64, 64])
Batch 18: torch.Size([32, 3, 64, 64])
Batch 19: torch.Size([32, 3, 64, 64])
Batch 20: torch.Size([32, 3, 64, 64])
Batch 21: torch.Size([32, 3, 64, 64])
Batch 22: torch.Size([32, 3, 64, 64])
Batch 23: torch.Size([32, 3, 64, 64])
Batch 24: torch.Size([32, 3, 64, 64])
Batch 25: torch.Size([32, 3, 64, 64])
Batch 26: torch.Size([

In [32]:
from torchvision.utils import save_image

with torch.no_grad():
    # Generate pure noise
    noise = torch.randn(1, 3, 64, 64).to(device)
    
    # Generate an image
    sampled = ddim_sample(model, noise, T=300)
    sampled = (sampled + 1) / 2  # scale from [-1, 1] to [0, 1]
    sampled = sampled.clamp(0, 1)
    
    # Save the generated image
    save_image(sampled, "results/final_ddim_sample.png")
    print("Generated image saved as: results/final_ddim_sample.png")


Generated image saved as: results/final_ddim_sample.png
